In [0]:
%run /Workspace/Users/rahulpatel@cyntexa.com/IAP_Practice/voltpath/00_setup

In [0]:
catalog = dbutils.widgets.get("catalog")

In [0]:
# F1 — Time travel: compare silver.chargers before batch 2 (version 2) vs current
# Batch 1 MERGE = version 1, CDF enabled = version 2, Batch 2 MERGE = version 3
# VERSION AS OF 2 gives the state after batch 1, before batch 2
charger_ids = "('CH004', 'CH011', 'CH027')"

spark.sql(f"""
CREATE OR REPLACE TABLE {catalog}.ops.time_travel_check AS
SELECT
    COALESCE(b.charger_id, c.charger_id) AS charger_id,
    b.max_kw        AS max_kw_before_batch2,
    c.max_kw        AS max_kw_current,
    b.is_active     AS is_active_before_batch2,
    c.is_active     AS is_active_current
FROM {catalog}.silver.chargers VERSION AS OF 2 b
FULL OUTER JOIN {catalog}.silver.chargers c
    ON b.charger_id = c.charger_id
WHERE b.charger_id IN {charger_ids}
   OR c.charger_id IN {charger_ids}
""")

# display(spark.sql(f"SELECT * FROM {catalog}.ops.time_travel_check ORDER BY charger_id"))

In [0]:
# F2 — Change Data Feed: capture all CDF rows from the batch 2 charger load (version 5)
# table_changes() SQL function returns 0 rows due to a Delta CDF limitation,
# so we use the PySpark DataFrame API with readChangeFeed option instead
cdf_df = (
    spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 5)
    .option("endingVersion", 5)
    .table(f"{catalog}.silver.chargers")
    .select(
        col("charger_id"),
        col("_change_type").alias("change_type"),
        col("max_kw"),
        col("is_active"),
        col("_commit_version").alias("commit_version"),
    )
)

cdf_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.ops.charger_changes_batch2")

# display(spark.sql(f"SELECT * FROM {catalog}.ops.charger_changes_batch2 ORDER BY charger_id, change_type"))

In [0]:
# F3 — Liquid clustering on gold.fact_sessions (date_key, station_id) and optimise
spark.sql(f"ALTER TABLE {catalog}.gold.fact_sessions CLUSTER BY (date_key, station_id)")
spark.sql(f"OPTIMIZE {catalog}.gold.fact_sessions")

# print("Liquid clustering applied and table optimised.")
# print("Clustering properties:")
# spark.sql(f"SHOW TBLPROPERTIES {catalog}.gold.fact_sessions").filter("key LIKE '%cluster%' OR key LIKE '%clustering%'").show(truncate=False)

In [0]:
# F4 — Deep clone silver.sessions, delete ST01 rows, undo with Delta RESTORE

# Step 1: Deep clone
spark.sql(f"CREATE OR REPLACE TABLE {catalog}.ops.sessions_drill DEEP CLONE {catalog}.silver.sessions")


# Disable auto-compaction to prevent concurrent write conflicts during RESTORE
spark.sql(f"ALTER TABLE {catalog}.ops.sessions_drill SET TBLPROPERTIES ('delta.autoOptimize.autoCompact' = 'false', 'delta.autoOptimize.optimizeWrite' = 'false')")

# Step 2: Capture version before delete (for later restore)
pre_delete_version = (
    spark.sql(f"DESCRIBE HISTORY {catalog}.ops.sessions_drill")
    .orderBy("version", ascending=False)
    .first()["version"]
)

original_count = spark.sql(f"SELECT COUNT(*) AS c FROM {catalog}.ops.sessions_drill").collect()[0][0]


# Step 3: Delete rows for station ST01 (via charger_id lookup)
spark.sql(f"""
DELETE FROM {catalog}.ops.sessions_drill
WHERE charger_id IN (SELECT charger_id FROM {catalog}.silver.chargers WHERE station_id = 'ST01')
""")
after_delete_count = spark.sql(f"SELECT COUNT(*) AS c FROM {catalog}.ops.sessions_drill").collect()[0][0]


# Step 4: Undo the delete with Delta RESTORE (not re-insert)
# Retry to handle concurrent auto-optimization conflicts
import time
for attempt in range(3):
    try:
        spark.sql(f"RESTORE TABLE {catalog}.ops.sessions_drill TO VERSION AS OF {pre_delete_version}")
        break
    except Exception as e:
        if "CONCURRENT_WRITE" in str(e) and attempt < 2:
            print(f"Restore attempt {attempt+1} failed (concurrent write), retrying...")
            time.sleep(3)
        else:
            raise
restored_count = spark.sql(f"SELECT COUNT(*) AS c FROM {catalog}.ops.sessions_drill").collect()[0][0]
print(f"Restore successful: {restored_count == original_count}")